In [ ]:
# Importamos las bibliotecas necesarias
import os
import pandas as pd    
import seaborn as sns  
import matplotlib.pyplot as plt  
import numpy as np
from scipy.spatial import distance
from sklearn.preprocessing import StandardScaler
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import fcluster
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import silhouette_samples
import warnings

In [ ]:
# Establecer el nivel de advertencias a "ignore" para ignorar todas las advertencias
warnings.filterwarnings("ignore")

In [ ]:
os.chdir(r'C:\Users\MAICKEL\OneDrive\Documents\MASTER_UCM\MODULO 7 -P2- MINERIA DE DATOS (CLUSTERS)\Documentación minería de datos y modelización predictiva  - Daniel Martín-20250121\Datos') 
datos = pd.read_excel('penguins.xlsx') 
datos = datos.set_index('species', drop=True)
df = datos.select_dtypes(include=[np.number]) # Seleccionar solo las columnas numéricas

# Visualizamos las primeras filas del DataFrame para verificar los datos
df.head()

# CALCULO DE DISTANCIAS

In [ ]:
# Calculamos distancias sin estandarizar
# Calcula la matriz de distancias Euclidianas entre las observaciones
distance_matrix = distance.cdist(df, df, 'euclidean')
# Crea un DataFrame a partir de la matriz de distancias con los índices de df
distance_df = pd.DataFrame(distance_matrix, index=df.index, columns=df.index)
# La distance_matrix es una matriz 2D que contiene las distancias Euclidianas 
# entre todas las parejas de observaciones.
# Seleccionamos las primeras 5 filas y columnas de la matriz de distancias
distance_small = distance_matrix[:5, :5]
# Agregamos los nombres de índice a la matriz de distancias
distance_small = pd.DataFrame(distance_small, index=df.index[:5], columns=df.index[:5])
# Redondeamos los valores en la matriz de distancias
distance_small_rounded = distance_small.round(2)
distance_small_rounded

In [ ]:
# Representamos gráficamente la matriz de distancias
# Crea una nueva figura para el gráfico con un tamaño específico
plt.figure(figsize=(10, 8))
# Genera un mapa de calor usando Seaborn
# - `distance_df`: DataFrame de pandas que contiene los datos para el mapa de calor.
# - `annot=False`: No muestra las anotaciones (valores de los datos) en las celdas del mapa.
# - `cmap="YlGnBu"`: Utiliza la paleta de colores "Yellow-Green-Blue" para el mapa de calor.
# - `fmt=".1f"`: Formato de los números en las anotaciones, en este caso no se usan.
sns.heatmap(distance_df, annot=False, cmap="YlGnBu", fmt=".1f")
# Muestra el gráfico
plt.show()

In [ ]:
# Realizamos clustering jerárquico para obtener la matriz de enlace (linkage matrix). 
# Clustermap es una función compleja que combina un mapa de calor con dendrogramas para mostrar la agrupación de datos.
# Aquí, estamos usando el dataframe 'distance_df' que contiene las distancias euclidianas entre las observaciones.
# La opción 'cmap' establece la paleta de colores a 'YlGnBu', que es un gradiente de azules y verdes.
# La opción 'fmt' se usa para formatear las anotaciones numéricas, en este caso a un decimal.
# La opción 'annot=False' indica que no queremos anotaciones numéricas en las celdas del mapa de calor.
# La opción 'method' especifica el método de agrupamiento a usar, en este caso 'average' que calcula la media de las distancias.
linkage = sns.clustermap(distance_df, cmap="YlGnBu", fmt=".1f", annot=False, method='ward').dendrogram_row


**Áreas oscuras (más azules)** representan distancias pequeñas, es decir, observaciones que están más cerca entre sí.

**Áreas más claras (más amarillas)** indican distancias más grandes, es decir, observaciones que son más diferentes entre sí.

Los brazos del dendrograma indican las relaciones jerárquicas entre las observaciones. Los grupos que están más cerca en el dendrograma tienen observaciones con distancias más pequeñas.

Este gráfico es útil para obtener una visión general de cómo las observaciones se agrupan según sus distancias y te proporciona una base sólida para profundizar en el análisis de clustering.


In [ ]:
# Estandarizamos los datos
# Inicializamos el escalador de estandarizacion
scaler = StandardScaler()
# Ajustamos y transformamos el DataFrame para estandarizar las columnas
# 'fit_transform' primero calcula la media y la desviacion estandar de cada columna para luego realizar la estandarizacion.
df_std = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# Asignamos el indice del DataFrame original 'df' al nuevo DataFrame 'df_std'
# Esto es importante para mantener la correspondencia de los indices de las filas despues de la estandarizacion.
df_std.index = df.index
df_std

In [ ]:
# Calculamos las distancias euclidianas por pares entre las filas del DataFrame estandarizado
# 'cdist' calcula la distancia euclidiana entre cada par de filas en 'df_std'.
# Esto resulta en una matriz de distancias donde cada elemento [i, j] es la distancia entre la fila i y la fila j.
distance_std = distance.cdist(df_std, df_std, 'euclidean') 
# Seleccionamos las primeras 10 filas y columnas de la matriz de distancias
distance_small = distance_std[:10, :10].round(2)
# Agregamos los nombres de índice a la matriz de distancias
distance_small = pd.DataFrame(distance_small, index=df.index[:10], columns=df.index[:10])

distance_small_rounded

In [ ]:
# Esto determina las dimensiones del grafico
plt.figure(figsize=(10, 8))
# Creamos un nuevo DataFrame para la matriz de distancias
# 'distance_std' se convierte en un DataFrame con indices y columnas correspondientes a 'df_std'
# Esto facilita la interpretacion del mapa de calor, ya que las filas y columnas estaran etiquetadas con los indices de 'df_std'
df_std_distance = pd.DataFrame(distance_std, index=df_std.index, columns=df_std.index)
# Generamos un mapa de calor utilizando Seaborn
# - 'df_std_distance': DataFrame que contiene los datos de distancia para visualizar.
# - 'annot=False': No muestra anotaciones numericas en cada celda del mapa de calor.
# - 'cmap="YlGnBu"': Define una paleta de colores en tonos de azul y verde para el mapa de calor.
# - 'fmt=".1f"': Formato de los numeros en las anotaciones, en este caso, un decimal.
sns.heatmap(df_std_distance, annot=False, cmap="YlGnBu", fmt=".1f")
# Mostramos el grafico resultante
plt.show()


En el gráfico anterior podemos observar, con ayuda de la matriz de distancias, que los valores tienen relación entre sí ya que cuando unos valores los tienen elevados, los otros suelen ser de un menor valor. Es decir, para un peso corporal de los pingüinos, si su pico es mas largo (bill_length_mm) su profundidad suele ser menor (bill_depth_mm).

# ANALISIS JERARQUICO

In [ ]:
# Realizamos clustering jerárquico para obtener la matriz de enlace (linkage matrix) sobre las distancias estandarizadas. 
linkage = sns.clustermap(df_std_distance, cmap="YlGnBu", fmt=".1f", annot=False, method='ward').dendrogram_row

En resumen, el gráfico representa un análisis jerárquico donde los pingüinos se agrupan en función de sus características físicas estandarizadas. El número óptimo de clusters parece ser 4 o 5, basándose en las divisiones visibles en el dendrograma. Las observaciones dentro de cada cluster son más similares entre sí, lo que indica que esos pingüinos comparten características físicas similares (masa corporal, longitud de los aletines, etc.).

In [ ]:
# Establecemos un umbral de color para el dendrograma
color_threshold = 10
linkage_matrix = sch.linkage(df_std, method='ward')  # Puedes elegir un metodo de enlace diferente si es necesario
# Creamos el dendrograma con el umbral de color especificado
dendrogram = sch.dendrogram(linkage_matrix, labels=df_std_distance.index.tolist(), leaf_rotation=90, color_threshold=color_threshold)
# Mostramos el dendrograma
plt.show()

El dendrograma muestra claramente 4 grupos de observaciones 
(representados por las diferentes ramas de colores naranja, verde, rojo, morado y marrón).

In [ ]:
# Asignamos las observaciones de datos a 4 clusters
# Especificamos el numero de clusters a formar
num_clusters = 4
# Asignamos los datos a los clusters
# 'fcluster' forma clusters planos a partir de la matriz de enlace 'linkage_matrix'
# 'criterion='maxclust'' significa que formaremos un numero maximo de 'num_clusters' clusters
cluster_assignments = fcluster(linkage_matrix, num_clusters, criterion='maxclust')
# Mostramos las asignaciones de clusters
print("Cluster Assignments:", cluster_assignments) 
# Creamos una nueva columna 'Cluster4' y asignamos los valores de 'cluster_assignments' a ella
# Ahora 'df' contiene una nueva columna 'Cluster4' con las asignaciones de cluster
df['Cluster4'] = cluster_assignments

In [ ]:
# Visualización de la distribución espacial de los clusters
# Paso 1: Realizar PCA
pca = PCA(n_components=2)  # Inicializamos PCA para 2 componentes principales
eliminar = ['Cluster4']
principal_components = pca.fit_transform(df.drop(eliminar, axis=1))  # Transformamos los datos a 2 componentes

fit = pca.fit(df_std)
# Calculamos las dos primeras componentes principales
resultados_pca = pd.DataFrame(fit.transform(df.drop(eliminar, axis=1)), 
                              columns=['Componente {}'.format(i) for i in range(1, fit.n_components_+1)],
                              index=df.index)

# Añadimos las componentes principales a la base de datos estandarizada.
df_z_cp = pd.concat([df_std, resultados_pca], axis=1)

# Calculo la matriz de correlaciones entre veriables y componentes
Correlaciones_var_comp = df_z_cp.corr()
Correlaciones_var_comp = Correlaciones_var_comp.iloc[:fit.n_features_in_, fit.n_features_in_:]
Correlaciones_var_comp

**Interpretación matriz correlaciones**

**El Componente 1** parece estar fuertemente influenciado por la masa corporal y la longitud del aletín, con una contribución moderada de la longitud del pico y una relación inversa con la profundidad del pico.

**El Componente 2** sigue siendo influenciado en gran parte por la masa corporal y la longitud del aletín, con una contribución moderada de la longitud del pico y la profundidad del pico de forma inversa.

In [ ]:
# Creamos un nuevo DataFrame para los componentes principales 2D
# Nos aseguramos de que df_pca tenga el mismo índice que df
df_pca = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'], index=df.index)

# Paso 2: Crear un gráfico de dispersión con colores para los clusters
plt.figure(figsize=(10, 6))  # Establecemos el tamaño del gráfico

# Recorremos las asignaciones únicas de clusters y trazamos puntos de datos con el mismo color
for cluster in np.unique(cluster_assignments):
    cluster_indices = df_pca.loc[cluster_assignments == cluster].index
    plt.scatter(df_pca.loc[cluster_indices, 'PC1'],
                df_pca.loc[cluster_indices, 'PC2'],
                label=f'Cluster {cluster}')  # Etiqueta para cada cluster
    # Anotamos cada punto con el nombre del país
    for i in cluster_indices:
        plt.annotate(i,
                     (df_pca.loc[i, 'PC1'], df_pca.loc[i, 'PC2']), fontsize=10,
                     textcoords="offset points",  # cómo posicionar el texto
                     xytext=(0,10),  # distancia del texto a los puntos (x,y)
                     ha='center')  # alineación horizontal puede ser izquierda, derecha o centro

# Líneas de referencia para los ejes x e y
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.axvline(0, color='black', linestyle='--', linewidth=0.5)

plt.title("Gráfico de PCA 2D con Asignaciones de Cluster")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.legend()
plt.grid()
plt.show()

Este gráfico permite ver cómo se agrupan las observaciones (pingüinos) en 4 clusters según sus características físicas, representadas en las dos primeras componentes principales del ACP. La agrupación indica que ciertos pingüinos tienen características similares en términos de peso corporal, tamaño de pico y aletín, mientras que otros están más distantes, mostrando diferencias notables en sus medidas.

In [ ]:
# Metodo del codo
# Creamos un array para almacenar los valores de WCSS para diferentes valores de K
wcss = []
    
for k in range(1, 11):  # Puedes elegir un rango diferente de valores de K
    kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
    kmeans.fit(df_std)
    wcss.append(kmeans.inertia_)  # Inserta es el valor de WCSS

# Graficamos los valores de WCSS frente al numero de grupos (K) y buscamos el punto "codo"
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='-', color='b')
plt.title('Metodo del Codo')
plt.xlabel('Numero de Clusters (K)')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

En el gráfico obtenido del método del codo, se observa que la WCSS disminuye drásticamente cuando el número de clusters K pasa de 1 a 2, y luego la disminución se vuelve más lenta y menos pronunciada a medida que K aumenta.

**Conclusión:**

El punto del codo se encuentra en K=3 o K=4, ya que después de esos puntos la reducción en la WCSS es menos pronunciada, lo que sugiere que agregar más clusters no mejora significativamente la calidad del agrupamiento.
K=3 o 4 parece ser el número óptimo de clusters, ya que es el punto en el que la mejora en la compactación de los grupos (medida por la WCSS) se desacelera.

In [ ]:
# Metodo de la silueta  
# Creamos un array para almacenar los puntajes de silueta para diferentes valores de K
silhouette_scores = []
    
# Ejecutamos el clustering K-means para un rango de valores de K y calculamos el puntaje de silueta para cada K
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=0)
    kmeans.fit(df_std)
    labels = kmeans.labels_
    silhouette_avg = silhouette_score(df_std, labels)
    silhouette_scores.append(silhouette_avg)
    
# Graficamos los puntajes de silueta frente al numero de clusters (K)
plt.figure(figsize=(8, 6))
plt.plot(range(2, 11), silhouette_scores, marker='o', linestyle='-', color='b')
plt.title('Metodo de la Silueta')
plt.xlabel('Numero de Clusters (K)')
plt.ylabel('Puntaje de Silueta')
plt.grid(True)
plt.show()


En K=2, el puntaje de silueta es alto (cerca de 0.5), lo que indica que los clusters generados estan bastante bien definidos y las observaciones están bien separadas.
A medida que K aumenta, el puntaje de silueta disminuye, alcanzando su valor más bajo en K=10 (aproximadamente 0.25). Esto sugiere que, al agregar más clusters, las observaciones no están siendo agrupadas de manera tan clara y los clusters se vuelven menos definidos.

**Conclusión:**

El valor óptimo de K basado en el puntaje de silueta parece ser K=2, ya que es cuando el puntaje de silueta es más alto. Esto indica que 2 clusters es probablemente el número más adecuado para este conjunto de datos, con clusters claramente separados.

# ANALISIS NO JERARQUICO

In [ ]:
# Configurar el número de clusters (k=4)
k = 4
# Inicializar el modelo KMeans
# 'n_clusters=k' indica el número de clusters a formar
# 'random_state=0' asegura la reproducibilidad de los resultados
# 'n_init=10' indica el número de veces que el algoritmo se ejecutará con diferentes centroides iniciales
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
# Ajustar el modelo KMeans a los datos estandarizados
# 'df_std' es el DataFrame que contiene los datos previamente estandarizados
kmeans.fit(df_std)
# Obtener las etiquetas de los clusters para los datos
# 'kmeans.labels_' contiene la asignación de cada punto a un cluster
kmeans_cluster_labels = kmeans.labels_
# Creamos una nueva columna 'Cluster' y asignamos los valores de 'kmeans_cluster_labels' a ella
# 'Cluster4_v2' sera el nombre de la nueva columna en el DataFrame 'df'
df['Cluster4_v2'] = kmeans_cluster_labels
# Ahora 'df' contiene una nueva columna 'Cluster4_v2' con las asignaciones de cluster
# Imprimimos los valores de la columna 'Cluster4_v2' para verificar las asignaciones de cluster
print(df["Cluster4_v2"])

In [ ]:
# Calculamos los valores de silueta para cada observación
silhouette_values = silhouette_samples(df_std, kmeans.labels_)
    
# Configuramos el tamaño de la figura para el gráfico
plt.figure(figsize=(8, 6))
y_lower = 10  # Inicio del margen inferior en el gráfico

# Iteramos sobre los 4 clusters para calcular los valores de silueta y dibujar el gráfico
for i in range(4):
    # Extraemos los valores de silueta para las observaciones en el cluster i
    ith_cluster_silhouette_values = silhouette_values[kmeans.labels_ == i]
    # Ordenamos los valores para que el gráfico sea más claro
    ith_cluster_silhouette_values.sort()
    
    # Calculamos donde terminarán las barras de silueta en el eje y
    size_cluster_i = ith_cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster_i
    
    # Elegimos un color para el cluster
    color = plt.cm.get_cmap("Spectral")(float(i) / 4)
    # Rellenamos el gráfico entre un rango en el eje y con los valores de silueta
    plt.fill_betweenx(np.arange(y_lower, y_upper),
                      0, ith_cluster_silhouette_values,
                      facecolor=color, edgecolor=color, alpha=0.7)
    # Etiquetamos las barras de silueta con el número de cluster en el eje y
    plt.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    # Actualizamos el margen inferior para el siguiente cluster
    y_lower = y_upper + 10  # 10 para el espacio entre clusters

# Títulos y etiquetas para el gráfico
plt.title("Gráfico de Silueta para los Clusters")
plt.xlabel("Valores del Coeficiente de Silueta")
plt.ylabel("Etiqueta del Cluster")
plt.grid(True)  # Añadimos una cuadrícula para mejor legibilidad
plt.show()  # Mostramos el gráfico resultante

Clasificación de observaciones: Todos los clusters muestran observaciones bien clasificadas, ya que sus coeficientes de silueta superan el 0.5 en todos los casos. Esto indica que las observaciones dentro de cada cluster están bien separadas de los demás clusters y adecuadamente agrupadas.

Cluster 2: Este cluster destaca por tener un mayor número de observaciones en comparación con los demás clusters, lo cual es evidente por su banda más ancha en el gráfico. Esto sugiere que el cluster 2 es más amplio, lo que podría indicar una mayor variabilidad interna entre las observaciones de este grupo.

In [ ]:
# Caracterizamos cada cluster
# Visualizacion de la distribucion espacial de los clusters
plt.figure(figsize=(10, 6))  # Definir el tamaño de la figura

# Iterar a traves de las etiquetas unicas de clusters y graficar puntos de datos con el mismo color
for cluster in np.unique(kmeans_cluster_labels):
    cluster_indices = df_pca.loc[kmeans_cluster_labels == cluster].index
    plt.scatter(df_pca.loc[cluster_indices, 'PC1'],
                df_pca.loc[cluster_indices, 'PC2'],
                label=f'Cluster {cluster}')  # Poner una etiqueta para cada cluster
    
    # Anotar cada punto con su respectivo indice
    for i in cluster_indices:
        plt.annotate(i,
                     (df_pca.loc[i, 'PC1'], df_pca.loc[i, 'PC2']),fontsize=10,
                     textcoords="offset points",  # Define como se posicionara el texto
                     xytext=(0,10),  # Define la distancia del texto a los puntos (x,y)
                     ha='center')  # Define la alineacion horizontal del texto

# Líneas de referencia para los ejes x e y
plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
plt.axvline(0, color='black', linestyle='--', linewidth=0.5)

# Configurar el titulo y las etiquetas de los ejes del grafico
plt.title("Grafico 2D de PCA con Asignaciones de Cluster KMeans")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.legend()  # Mostrar la leyenda
plt.grid()  # Mostrar la cuadricula
plt.show()  # Mostrar el grafico

**Distribución espacial de los clusters:**

El gráfico muestra la distribución espacial de las observaciones en el espacio de las dos primeras componentes principales (PC1 y PC2). 

**Cluster 0** (azul): Este cluster está concentrado principalmente en la parte inferior izquierda del gráfico, lo que indica que estas observaciones tienen valores más bajos en PC1 y PC2. Este cluster parece estar formado por pingüinos con características físicas distintivas que se agrupan en esta región.

**Cluster 1** (rojo): El cluster 1 está distribuido en la zona intermedia, más cerca del eje X (PC1) positivo. Esto podría indicar una relación con características que se agrupan principalmente a lo largo de PC1.

**Cluster 2** (verde): Este cluster está más disperso, con una concentración de puntos en la parte central y superior izquierda. Es probable que las observaciones en este grupo sean más heterogéneas en cuanto a sus características.

**Cluster 3** (naranja): El cluster 3 está claramente situado en la parte superior derecha del gráfico, lo que sugiere que estas observaciones tienen valores altos tanto en PC1 como en PC2. Este cluster probablemente contiene las observaciones con características físicas significativamente diferentes.

In [ ]:
# Caracterizamos cada cluster
# Añadimos las etiquetas como una nueva columna al DataFrame original
df['Cluster'] = kmeans.labels_
# Ordenamos el DataFrame por la columna "Cluster"
df_sort = df.sort_values(by="Cluster")

# Agrupamos los datos por la columna 'Cluster' y calculamos la media de cada grupo
# Esto proporcionará las coordenadas de los centroides de los clusters en el espacio de los datos originales
cluster_centroids_orig = df_sort.groupby('Cluster').mean()
# Redondeamos los valores para facilitar la visualización
cluster_centroids_orig.round(2)
# 'cluster_centroids_orig' ahora contiene los centroides de cada cluster en el espacio de los datos originales

**Resumen de las características clave según cluster**

**Cluster 0:** Picos largos y más profundos, aletas cortas, peso corporal bajo.

**Cluster 1:** Picos más largos, aletas más largas, peso corporal alto.

**Cluster 2:** Picos más cortos y bastante gruesos, aletas más cortas, peso corporal más bajo.

**Cluster 3:** Picos intermedios y los más delgados, aletas largas, peso corporal intermedio.